# Benchmark darija de Wakil, sur un GPU Google Colab

Ce notebook fait passer le benchmark de Wakil (`evals/benchmark`) aux modèles de `promptfooconfig.yaml`, sur un GPU
gratuit de Colab au lieu du PC. **Données 100 % fictives – projet de démonstration.**

**Avant de commencer**
1. Pousser sur GitHub les derniers changements (questions, code) : ce notebook clone le dépôt, pas le PC.
2. Dans Colab : *Exécution → Modifier le type d'exécution → GPU T4*.
3. Exécuter les cellules dans l'ordre (*Exécution → Tout exécuter*).

**Ce qu'il faut savoir**
- Aucun secret ni base de données : le benchmark n'a besoin que d'Ollama, des deux serveurs MCP (la liste des outils)
  et de promptfoo. Aucun outil n'est exécuté.
- Les modèles sont retéléchargés à chaque session (environ 14 Go, sur le réseau de Google).
- **Ne pas mélanger ces résultats avec ceux du PC** : un GPU calcule très légèrement autrement qu'un processeur.
  Tous les modèles comparés doivent passer sur la même machine. Les temps de réponse sont ceux d'un GPU T4.

## 1. Réglages

In [ ]:
DEPOT = "https://github.com/Oumaima-Lg/wakil.git"
BRANCHE = "main"
FILTRE = ""  # "" : tous les modèles de promptfooconfig.yaml ; « qwen » : seulement ceux dont le nom contient « qwen »

## 2. Le GPU est-il là ?

In [ ]:
import subprocess

sortie = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if sortie.returncode != 0:
    raise SystemExit("Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4, puis relancer.")
print(sortie.stdout)

## 3. Le dépôt

In [ ]:
%cd /content
!rm -rf wakil && git clone --quiet --depth 1 --branch {BRANCHE} {DEPOT} wakil
!cd wakil && git log --oneline -1

## 4. Ollama et les modèles

Les modèles à télécharger sont lus dans `promptfooconfig.yaml` : la même liste que celle du benchmark.

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import time
import urllib.request

ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except OSError:
        time.sleep(1)
else:
    raise SystemExit("Ollama ne démarre pas : voir /content/ollama.log")
print("Ollama répond.")

In [ ]:
import yaml

config = yaml.safe_load(open("/content/wakil/evals/benchmark/promptfooconfig.yaml"))
modeles = [f["config"]["modele"] for f in config["providers"] if FILTRE in f["label"]]
print("Modèles :", modeles)
for modele in modeles:
    print(f"\n=== {modele}")
    subprocess.run(["ollama", "pull", modele], check=True, stdout=subprocess.DEVNULL)
!ollama list

## 5. Les dépendances

uv installe Python 3.14 et les paquets des trois services utiles ; Node 22 sert à promptfoo.

In [ ]:
!pip install -q uv
for service in ("agent-api", "mcp-tools", "rag-faq"):
    print("===", service)
    subprocess.run(["uv", "sync", "--quiet"], cwd=f"/content/wakil/apps/{service}", check=True)

!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get -qq install -y nodejs > /dev/null
!node --version
!cd /content/wakil/evals && npm ci --silent --no-audit --no-fund

## 6. Les deux serveurs MCP

Ils annoncent les outils de Wakil et leurs descriptions (sans base de données : aucun outil n'est exécuté).

In [ ]:
import socket

serveurs = {}
for service, port in (("mcp-tools", 8002), ("rag-faq", 8004)):
    serveurs[service] = subprocess.Popen(
        ["uv", "run", "uvicorn", "app.serveur:app", "--port", str(port)],
        cwd=f"/content/wakil/apps/{service}",
        stdout=open(f"/content/{service}.log", "w"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                break
        time.sleep(1)
    else:
        raise SystemExit(f"{service} ne démarre pas : voir /content/{service}.log")
    print(service, "répond sur le port", port)

## 7. Vérification avant la mesure

Les outils que le LLM verra, annoncés par les serveurs MCP : ils doivent tous être là.

In [ ]:
!cd /content/wakil/evals/benchmark && ../../apps/agent-api/.venv/bin/python -c "import fournisseur; print(sorted(d['function']['name'] for d in fournisseur._descriptions()))"

## 8. La mesure

Un modèle à la fois, puis le rapport (`resultats.md`).

In [ ]:
!cd /content/wakil/evals && BENCHMARK_MACHINE="Google Colab, GPU $(nvidia-smi --query-gpu=name --format=csv,noheader)" npm run benchmark -- {FILTRE}


## 9. Les résultats

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("/content/wakil/evals/benchmark/resultats.md", encoding="utf-8").read()))

In [ ]:
import shutil

from google.colab import files

shutil.copy("/content/wakil/evals/benchmark/resultats.md", "/content/wakil/evals/benchmark/resultats/resultats.md")
archive = shutil.make_archive("/content/resultats-benchmark", "zip", "/content/wakil/evals/benchmark/resultats")
files.download(archive)  # resultats.md et les réponses brutes de chaque modèle (JSON)